In [ ]:
# Importing Libraries
import pandas as pd
import ee
import geemap
import numpy as np

In [ ]:
# Initialize Google Earth Engine API with the registered cloud project
ee.Initialize(project ='tea-yield-ml-project') 

In [ ]:
# Define spatial coordinates and create Earth Engine Point geometry
# Replace with your own coordinates
LATITUDE = 7.2800 
LONGITUDE= 80.6000
point = ee.Geometry.Point([longitude,latitude])

In [ ]:
# Load primary yield and weather dataset, preprocess dates, and extract temporal range
main_df=pd.read_csv(r"Tea Yield data collection_with_weather _1Cleaned.csv")
main_df['plucking_date']=pd.to_datetime(main_df['plucking_date'])
main_df= main_df.drop(columns='NDVI')

# Set dynamic start and end dates based on the dataset's date range
START_DATE = main_df['plucking_date'].min().strftime('%Y-%m-%d')
END_DATE = main_df['plucking_date'].max().strftime('%Y-%m-%d')

In [ ]:
# Helper function to mask clouds and shadows using Sentinel-2 Scene Classification Layer (SCL)
def mask_s2_clouds(image):
    scl= image.select('SCL')
    bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
    mask = bad.Not()
    return(image.updateMask(mask)
                .divide(1000)
                .copyProperties(image,['system:time_start']))







In [ ]:
# Helper function to calculate Normalized Difference Vegetation Index (NDVI) band
def add_ndvi(image):
    ndvi= image.normalizedDifference(['B8','B4']).rename('NDVI')
    return image.addBands(ndvi)




In [ ]:
# Query, filter, and process Sentinel-2 Surface Reflectance (S2_SR_HARMONIZED) image collection
collection = ( ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterDate(START_DATE,END_DATE)
              .filterBounds(point)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',70))
              .map(mask_s2_clouds)
              .map(add_ndvi)
)

In [ ]:
# Helper function to reduce region and extract mean NDVI value for the target point geometry
def extract_ndvi(image):
    stat=image.select('NDVI').reduceRegion(
        reducer = ee.Reducer.mean(),
        geometry= point, scale = 10, maxPixels =1e9,bestEffort=True 
    )
    return ee.Feature(None,{ 'date': image.date().format('YYYY-MM-dd'),
                              'NDVI':stat.get('NDVI')
                            })


In [ ]:
# Map extraction function across collection and filter out null NDVI features
ndvi_fc = collection.map(extract_ndvi).filter(ee.Filter.notNull(['NDVI']))

In [ ]:
# Convert Earth Engine FeatureCollection to a server-side list and download client-side
ndvi_list = ndvi_fc.reduceColumns(
    ee.Reducer.toList(2),['date','NDVI']
).get('list').getInfo()

In [ ]:
# Construct DataFrame from satellite NDVI data and perform linear interpolation to align dates
df=pd.DataFrame(ndvi_list,columns=['Date','NDVI'])
df['Date']=pd.to_datetime(df['Date'])
df=df.sort_values('Date').drop_duplicates(subset='Date')

# Interpolate satellite NDVI values to match exact plucking dates in the main dataset
main_df['NDVI']=np.interp(
    main_df['plucking_date'].astype('int'),
    df['Date'].astype('int64'),
    df['NDVI']

)



In [ ]:
# Display updated dataset with integrated NDVI values
main_df

,Year,Month,Day,plucking_date,Total yield weight per cycle (kg),Rainfall,Avg Temp,Relative humidity,plucking_interval,total_rain_mm,NDVI
0,2020,10,5,2020-10-05,4.0,0.80,26.40,82.00,0,0.8,0.806887
1,2020,10,17,2020-10-17,2.0,6.06,26.26,86.00,12,72.7,0.804064
2,2020,11,2,2020-11-02,2.0,3.09,26.24,86.75,16,49.4,0.799548
3,2020,11,9,2020-11-09,4.0,31.16,25.83,88.29,7,218.1,0.797573
4,2020,11,20,2020-11-20,1.0,31.78,25.52,90.00,11,349.6,0.794468
...,...,...,...,...,...,...,...,...,...,...,...
174,2026,7,21,2026-07-21,21.0,2.93,27.19,82.47,15,43.9,0.783425
175,2026,8,1,2026-08-01,18.0,8.03,27.04,82.64,11,88.3,0.782295
176,2026,8,13,2026-08-13,17.0,10.67,26.03,86.83,12,128.0,0.781063
177,2026,8,28,2026-08-28,27.0,8.49,26.22,86.40,15,127.4,0.780139


In [ ]:
# # Save final dataset to a CSV file
main_df.to_csv("E:\DATA ANALYST\Tea Yield final dataset.csv")